In [1]:
import sys
from pathlib import Path

from src.post_processing.main import tokens_to_html_after_decomposed1_3_prompting
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_DIR, FEWSHOT_CACHE_DIR, PROMPT_DIR
import json
from src.extractor import LabelTransformConfig, prepare_label_tokens, _parse_parent_annotations
    
from src.tokenizer_utils import tokenize, decode
from src.htmlLabel import simplified_to_normal_form


c:\Users\zakga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Choose the prompting configuration

In [2]:
filename = "1989CanLII1415ONCA"
split = "test"
filepath = Path(DATA_DIR) / "original" / split / f"{filename}.html"
#filepath = Path("output") / f"{filename}_processed2.html"
with open(filepath, "r", encoding="utf-8") as f:
    html_content = f.read()

In [3]:
### Choose the right worflow
method = "AIO" # "AIO" | "DEC0" | "DEC1" | "DEC2" | "DEC3"

if method == "AIO":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = []
    new_labels = ["decision", "legislation", "secondary sources", "title", "citation", "source", "authors", "fragment"]

    spans_in_context = True

    prompt_filename = "allInOne_long.txt"


if method == "DEC0":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = []
    new_labels = ["decision", "legislation", "secondary sources"]

    spans_in_context = True

    prompt_filename = "decomposed0_long.txt"

if method == "DEC1":
    parents = ["decision", "legislation", "secondary sources"]
    already_labeled_labels = ["decision", "legislation", "secondary sources"]
    new_labels = ["title", "fragment"]

    spans_in_context = False


    prompt_filename = "decomposed1-3.txt"

if method == "DEC2":
    parents = ["secondary sources"]
    already_labeled_labels = ["decision", "legislation", "secondary sources", "title", "fragment"]
    new_labels = ["source", "authors"]

    spans_in_context = False

    prompt_filename = "decomposed1-3.txt"

if method == "DEC3":
    parents = ["decision", "legislation"]
    already_labeled_labels = ["decision", "legislation", "secondary sources", "title", "fragment", "source", "authors"]
    new_labels = ["citation"]

    spans_in_context = False

    prompt_filename = "decomposed1-3.txt"

#### Commun Few SHot Selection

In [4]:
fewshot_method = "greedy"   # "greedy" | "random"

with open(FEWSHOT_CACHE_DIR / f"examples_{fewshot_method}.json", "r", encoding="utf-8") as f:
    fewshot_file_content = json.load(f)

fewshot_examples = [(example["example"]["input"], example["example"]["output"]) for example in fewshot_file_content["examples"]]



##### Few Shot processing step

In [5]:
nb_fewshot_examples = 6
allowed_labels = already_labeled_labels + new_labels

input_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_labels=already_labeled_labels,
    keep_attributes=["labelname"]
)

output_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_labels=already_labeled_labels + new_labels,
    keep_attributes=["labelname"]
)


# Transform the output in it simplified form
final_fewshot = []
total_output_text = ""
for example in fewshot_examples:
    input, output = example

    input_tokens = tokenize(input)
    transformed_input_tokens = prepare_label_tokens(input_tokens, input_label_config)

    output_tokens = tokenize(output)
    transformed_output_tokens = prepare_label_tokens(output_tokens, output_label_config)

    if spans_in_context:
        final_fewshot.append((decode(transformed_input_tokens), decode(transformed_output_tokens)))

    if not spans_in_context:

        total_output_text += "|||" + decode(transformed_output_tokens)

final_fewshot = final_fewshot[:nb_fewshot_examples]


if not spans_in_context:
    parents_dict = _parse_parent_annotations(total_output_text)
    for parent_name, annotations in parents_dict.items():
        if parent_name not in parents:
            continue
        for annotation in annotations:
            input = decode(prepare_label_tokens(simplified_to_normal_form(tokenize(annotation),label_type="manual_label"), input_label_config))

            if input != annotation:
                final_fewshot.append((input, annotation))

#### Commun Prompt loading

In [6]:
from src.prompts.prompt_utils import build_sublabel_definitions

with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)

from src.prompts.sublabel_definitions import SUBLABEL_DEFINITIONS_V2


if method in ["DEC1", "DEC2", "DEC3"]:
    sublabels_str = ", ".join(new_labels)
    sublabels_definition = build_sublabel_definitions(set(new_labels) - set(parents), sublabel_definitions=SUBLABEL_DEFINITIONS_V2)

    system_prompt = system_prompt.format(
            sublabels=sublabels_str,
            sublabels_definition=sublabels_definition,
        )

system_prompt used :  allInOne_long.txt


#### Assistant loading

In [7]:
from src.models import AssistantFactory

gpt5_2_config= {
        "type": "openai",
        "model_name": "gpt-5.2",
        "temperature": 1,
    }

assistant = AssistantFactory.create_from_config(gpt5_2_config)

#### Chunk output controle

In [8]:
def process_output(generated, token, allowed_labels, assistant, with_fallback: bool = True):

    from src.output_control.processor import OutputProcessor
    from src.output_control.fallback import FallbackHandler 

    controller = OutputProcessor()
    fallback_handler = FallbackHandler(processor=controller)
    corrected_generated_tokens, status = controller.process(raw_llm_output=generated,
            original_chunk=token,
            allowed_labels=allowed_labels)
    
    if not status.passed and not with_fallback:
        return token, status

    if not status.passed and with_fallback:
            print("Output did not pass verification. Invoking fallback mechanism...")
            corrected_generated_tokens, status =  fallback_handler.handle_failure(
                    assistant=assistant,
                    corrected_output=corrected_generated_tokens,
                    original_chunk=token,
                    initial_status=status,
                    allowed_labels=allowed_labels,
                    fallback_prompt_filename= "fallback.txt"
                    )
            
    return corrected_generated_tokens, status

### For AIO or DEC0 ONLY

#### Chunking with the chunker

In [9]:
chunker = "paragraph"  # "paragraph" | "sentence"

from src.chunkers.cache import cache_exists, load_cache
from src.chunkers import ChunkerFactory

if not cache_exists(chunker, split, filename):

    # Load spaCy only if needed
    nlp = None
    if chunker == "sentence":
        import spacy
        nlp = spacy.load("en_core_web_trf")
        print("✅ Model loaded.\n")


    token_chunks = ChunkerFactory.get_chunks(
        html_content, method=chunker, split=split, filename=filename, nlp=nlp
    )

else:
    token_chunks = load_cache(chunker, split, filename)




#### Main processing function

In [10]:
from src.models import get_message
from tqdm import tqdm

processed_chunks = []
for token_chunk in tqdm(token_chunks):

    user_input =  decode(token_chunk)

    message = get_message(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=True)

    generated = assistant.generate(message=message)

    corrected_generated_tokens, status = process_output(generated, token=token_chunk, allowed_labels=allowed_labels, assistant=assistant)


    processed_chunks.append(corrected_generated_tokens)

100%|██████████| 18/18 [01:31<00:00,  5.07s/it]


#### Post Processing

In [12]:
from src.post_processing import chunks_to_html

output_html_content = chunks_to_html(processed_chunks, html_content)

   ✓ Flattened 18 chunks into 10113 tokens
   ✓ Merged to 12453 tokens
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)
   ✓ Corrected 14029 tokens
   ✓ Brackets are coherent

✓ POST-PROCESSING COMPLETE
Final HTML length: 144013 characters



#### File saving

In [13]:
output_filename = Path("output") / f"{filename}_processed.html"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(output_html_content)

### For DEC1-3 

No chunking needed here, we just need the list of mention already labeled

#### Convert into tokens

In [9]:
from src import extract_body, tokenize, clean_tokens
tokens = tokenize(html_content)


#### Get already extracted mention

In [ ]:

from src.extractor import build_processing_segments
from src.extractor import get_list_of_mention
from src.models import get_message
from tqdm import tqdm
from src.models import get_message
from tqdm import tqdm

parent_mentions = get_list_of_mention(
        tokens=tokens,
        keep_labels=parents,
        label_type="auto_label"  # Process auto_labels from parent extraction
    )

print(f"Found {len(parent_mentions)} parent mentions to process")

segments = build_processing_segments(tokens, parent_mentions)

print(f"Built {len(segments)} token segments "
        f"({sum(s['process'] for s in segments)} to process)")

Found 50 parent mentions to process
Built 101 token segments (50 to process)


#### Main processing function

In [ ]:
config = LabelTransformConfig(
    use_simplified=False,
    switch_type=False,
    keep_labels=already_labeled_labels,
    keep_attributes=["labelname"]
) # We only remove the attribute
 

failed_count = 0

for idx, segment in enumerate(tqdm(segments, desc="Processing mentions")):
        if not segment["process"]:
            continue


        mention = segment["tokens"]
        html_label = segment["meta"]["label"]

        
        prepared_tokens = _prepare_label_tokens(mention, input_label_config)
        user_input = decode(prepared_tokens)

        filtered_fewshot = []
        for example in final_fewshot:
            if example[0].startswith(f"<{html_label.name}>"):
                filtered_fewshot.append(example)
        message = get_message(system_prompt=system_prompt, user_input=user_input, fewshot_examples=filtered_fewshot, has_system_role=True)


        generated = assistant.generate(message=message)

        
        corrected_generated_tokens, status = process_output(generated=generated, token=_prepare_label_tokens(mention, config), allowed_labels=allowed_labels, assistant=assistant, with_fallback=False)
        
        if not status.passed:
            failed_count += 1

        segment["tokens"] = corrected_generated_tokens

processed_tokens = [
        token
        for segment in segments
        for token in segment["tokens"]
    ]

#### Post Processing : tokens to HTML

In [ ]:
from src.post_processing.main import tokens_to_html_after_decomposed1_3_prompting
processed_html_content = tokens_to_html_after_decomposed1_3_prompting(processed_tokens, html_content)

   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)


#### Save File

In [16]:
output_filename = Path("output") / f"{filename}_processed2.html"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(processed_html_content)